# Chapter 5.3: RadixAttention — Radix Tree for KV-Cache

## Learning Objectives

By the end of this notebook, you will:

1. **Understand the RadixAttention paper** and its key contributions
2. **Master the radix tree (Patricia trie) data structure**
3. **See how RadixAttention stores and reuses KV-cache** across requests
4. **Compare RadixAttention with PagedAttention** for prefix sharing
5. **Implement a RadixAttention simulator** from scratch
6. **Visualize cache hit scenarios** for common use cases

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part5/chapter_5.3_radix_attention.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part5/chapter_5.3_radix_attention.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

## 1. The Problem: KV-Cache Reuse

### Why Prefix Caching Matters

In LLM serving, many requests share **common prefixes**:

- **System prompts**: All chat requests with the same system prompt
- **Multi-turn conversations**: Each turn shares all previous turns
- **Few-shot examples**: Many requests share the same examples
- **Document QA**: Multiple questions about the same document

Without prefix caching, we recompute KV-cache for shared prefixes repeatedly:

```
Request 1: [System prompt | User Q1] -> compute all KV-cache
Request 2: [System prompt | User Q2] -> compute all KV-cache again!
Request 3: [System prompt | User Q3] -> compute all KV-cache again!

Wasted computation: System prompt KV-cache computed 3x!
```

### The RadixAttention Solution

RadixAttention uses a **radix tree** to automatically detect and share prefixes:

```
Request 1: [System prompt | User Q1] -> compute all, store in tree
Request 2: [System prompt | User Q2] -> reuse system prompt KV-cache!
Request 3: [System prompt | User Q3] -> reuse system prompt KV-cache!

Saved: 2x system prompt computation
```

## 2. Radix Tree (Patricia Trie) Data Structure

### 2.1 What is a Radix Tree?

A radix tree (also called a **Patricia trie** or **compressed trie**) is a space-optimized trie where:
- Each internal node with only one child is **merged with its child**
- Edges can contain **sequences** (not just single characters/tokens)

```
Regular Trie:              Radix Tree (compressed):

      root                       root
      / \                        / \
     t   a                    test  aster
     |   |
     e   s
     |   |
     s   t
     |   |
     t   e
         |
         r
```

### 2.2 Why Radix Tree for KV-Cache?

1. **Automatic prefix detection**: Shared prefixes are naturally shared in the tree
2. **Variable-length edges**: Can store sequences of tokens, not just individual tokens
3. **Efficient insertion**: O(n) where n is the length of the new sequence
4. **LRU eviction**: Can evict least-recently-used leaf nodes when memory is full
5. **No hash collisions**: Unlike hash-based prefix caching

In [ ]:
import matplotlib.pyplot as pltimport matplotlib.patches as mpatchesimport numpy as np# Consistent color palette for all diagramsBLUE = '#4A90D9'GREEN = '#27AE60'ORANGE = '#F39C12'RED = '#E74C3C'PURPLE = '#8E44AD'LIGHT_BLUE = '#85C1E9'LIGHT_GREEN = '#82E0AA'LIGHT_ORANGE = '#F8C471'LIGHT_RED = '#F1948A'LIGHT_PURPLE = '#C39BD3'GRAY = '#95A5A6'DARK_GRAY = '#2C3E50'fig, ax = plt.subplots(1, 1, figsize=(14, 9))ax.set_xlim(-1, 15)ax.set_ylim(-1, 10)ax.axis('off')fig.patch.set_facecolor('white')# Titleax.text(7, 9.5, 'Radix Tree for KV-Cache: Prefix Sharing',        ha='center', fontsize=14, fontweight='bold', color=DARK_GRAY)# Root nodedef draw_node(ax, x, y, text, color, width=2.8, height=0.7):    box = mpatches.FancyBboxPatch((x - width/2, y - height/2), width, height,                                   boxstyle="round,pad=0.1", facecolor=color,                                   edgecolor='white', linewidth=2, alpha=0.9)    ax.add_patch(box)    ax.text(x, y, text, ha='center', va='center', fontsize=8,            fontweight='bold', color='white', wrap=True)def draw_edge(ax, x1, y1, x2, y2):    ax.annotate('', xy=(x2, y2+0.35), xytext=(x1, y1-0.35),                arrowprops=dict(arrowstyle='->', color=GRAY, lw=1.5))# Rootdraw_node(ax, 7, 8, "[ROOT]", DARK_GRAY, 1.5, 0.6)# System prompt (shared by all 3 requests)draw_node(ax, 7, 6.5, '"You are a helpful\nassistant..." (50 tokens)', BLUE, 3.5, 0.9)draw_edge(ax, 7, 8, 7, 6.5)# Shared prefix annotationax.annotate('Shared prefix\n(computed ONCE)', xy=(3.2, 6.5), fontsize=9,            fontweight='bold', color=BLUE, ha='center',            bbox=dict(boxstyle='round,pad=0.3', facecolor=LIGHT_BLUE, alpha=0.3))ax.annotate('', xy=(5.2, 6.5), xytext=(4.2, 6.5),            arrowprops=dict(arrowstyle='->', color=BLUE, lw=1.5))# Three user messages branching outdraw_node(ax, 2.5, 4.5, 'User: "What is AI?"\n(5 tokens)', GREEN, 2.8, 0.9)draw_node(ax, 7, 4.5, 'User: "Explain Python"\n(6 tokens)', ORANGE, 2.8, 0.9)draw_node(ax, 11.5, 4.5, 'User: "Write a poem"\n(8 tokens)', RED, 2.8, 0.9)draw_edge(ax, 5.5, 6.5, 2.5, 4.5)draw_edge(ax, 7, 6.5, 7, 4.5)draw_edge(ax, 8.5, 6.5, 11.5, 4.5)# Generated responsesdraw_node(ax, 2.5, 2.5, 'Response tokens\n(20 tokens)', LIGHT_GREEN, 2.5, 0.8)draw_node(ax, 7, 2.5, 'Response tokens\n(30 tokens)', LIGHT_ORANGE, 2.5, 0.8)draw_node(ax, 11.5, 2.5, 'Response tokens\n(15 tokens)', LIGHT_RED, 2.5, 0.8)draw_edge(ax, 2.5, 4.5, 2.5, 2.5)draw_edge(ax, 7, 4.5, 7, 2.5)draw_edge(ax, 11.5, 4.5, 11.5, 2.5)# Request labelsax.text(2.5, 1.3, 'Request 1', ha='center', fontsize=10, fontweight='bold', color=GREEN)ax.text(7, 1.3, 'Request 2', ha='center', fontsize=10, fontweight='bold', color=ORANGE)ax.text(11.5, 1.3, 'Request 3', ha='center', fontsize=10, fontweight='bold', color=RED)# Cache hit annotationax.text(7, 0.3, 'Cache hit rate: 50/(50+5) = 91% for Request 1  |  50/(50+6) = 89% for Request 2  |  50/(50+8) = 86% for Request 3',        ha='center', fontsize=9, color=DARK_GRAY,        bbox=dict(boxstyle='round,pad=0.4', facecolor='#F0F0F0', alpha=0.8))plt.tight_layout()plt.savefig("radix_tree_prefix_sharing.png", dpi=150, bbox_inches='tight', facecolor='white')plt.show()

In [ ]:
# Basic Radix Tree implementation for understanding

from typing import Dict, List, Optional, Tuple
import time

class RadixNode:
    """A node in the radix tree.
    
    Each node represents a segment of token IDs.
    The node stores:
    - token_ids: the sequence of tokens on the edge leading to this node
    - children: child nodes (keyed by first token of child's edge)
    - kv_indices: indices into the KV-cache pool for this segment
    - ref_count: number of active requests using this node
    - last_access: timestamp for LRU eviction
    """
    
    def __init__(self, token_ids: Tuple[int, ...] = (), parent=None):
        self.token_ids = token_ids       # Tokens on edge to this node
        self.children: Dict[int, 'RadixNode'] = {}  # first_token -> child
        self.parent = parent
        self.kv_indices: List[int] = []  # Indices in KV-cache pool
        self.ref_count = 0               # Active references
        self.last_access = time.time()   # For LRU eviction
    
    @property
    def num_tokens(self):
        return len(self.token_ids)
    
    @property
    def is_leaf(self):
        return len(self.children) == 0
    
    def __repr__(self):
        return f"RadixNode(tokens={self.token_ids}, children={len(self.children)}, refs={self.ref_count})"


class SimpleRadixTree:
    """Simplified radix tree for KV-cache management.
    
    This demonstrates the core RadixAttention algorithm.
    """
    
    def __init__(self):
        self.root = RadixNode()
        self.total_nodes = 1
    
    def match_prefix(self, token_ids: List[int]) -> Tuple[RadixNode, int]:
        """Find the longest prefix match in the tree.
        
        Returns:
            (last_matched_node, num_matched_tokens)
        """
        node = self.root
        matched = 0
        pos = 0
        
        while pos < len(token_ids):
            first_token = token_ids[pos]
            
            if first_token not in node.children:
                break  # No matching child
            
            child = node.children[first_token]
            edge_tokens = child.token_ids
            
            # Compare edge tokens with remaining input
            match_len = 0
            for i in range(min(len(edge_tokens), len(token_ids) - pos)):
                if edge_tokens[i] == token_ids[pos + i]:
                    match_len += 1
                else:
                    break
            
            if match_len == 0:
                break  # Edge doesn't match
            
            if match_len < len(edge_tokens):
                # Partial edge match - we matched some but not all
                matched += match_len
                break
            
            # Full edge match
            node = child
            matched += match_len
            pos += match_len
            node.last_access = time.time()
        
        return node, matched
    
    def insert(self, token_ids: List[int], kv_indices: List[int] = None):
        """Insert a token sequence into the tree.
        
        If the sequence shares a prefix with existing entries,
        the tree is split at the divergence point.
        """
        node = self.root
        pos = 0
        
        while pos < len(token_ids):
            first_token = token_ids[pos]
            
            if first_token not in node.children:
                # No matching child — create new node
                remaining = tuple(token_ids[pos:])
                new_node = RadixNode(remaining, parent=node)
                if kv_indices:
                    new_node.kv_indices = kv_indices[pos:]
                node.children[first_token] = new_node
                self.total_nodes += 1
                return new_node
            
            child = node.children[first_token]
            edge_tokens = child.token_ids
            
            # Find where the edge and input diverge
            match_len = 0
            for i in range(min(len(edge_tokens), len(token_ids) - pos)):
                if edge_tokens[i] == token_ids[pos + i]:
                    match_len += 1
                else:
                    break
            
            if match_len < len(edge_tokens):
                # Partial match — split the edge
                # Before: node -> child("abcde")
                # After:  node -> split("abc") -> child("de")
                #                             -> new("fg")
                
                split_node = RadixNode(
                    tuple(edge_tokens[:match_len]), parent=node
                )
                split_node.kv_indices = child.kv_indices[:match_len]
                
                # Update child to be the remainder
                child.token_ids = tuple(edge_tokens[match_len:])
                child.kv_indices = child.kv_indices[match_len:]
                child.parent = split_node
                
                # Link split node
                split_node.children[edge_tokens[match_len]] = child
                node.children[first_token] = split_node
                
                self.total_nodes += 1
                
                # Add remaining input as new child of split
                if pos + match_len < len(token_ids):
                    remaining = tuple(token_ids[pos + match_len:])
                    new_node = RadixNode(remaining, parent=split_node)
                    if kv_indices:
                        new_node.kv_indices = kv_indices[pos + match_len:]
                    split_node.children[token_ids[pos + match_len]] = new_node
                    self.total_nodes += 1
                    return new_node
                return split_node
            
            # Full match — continue down
            node = child
            pos += match_len
        
        return node
    
    def visualize(self, node=None, prefix="", is_last=True, depth=0):
        """Print tree structure."""
        if node is None:
            node = self.root
            print("RadixTree Visualization")
            print("=" * 50)
            print("[root]")
        
        children = list(node.children.values())
        for i, child in enumerate(children):
            is_child_last = (i == len(children) - 1)
            connector = "└── " if is_child_last else "├── "
            tokens_str = str(list(child.token_ids))
            if len(tokens_str) > 30:
                tokens_str = tokens_str[:27] + "..."
            ref_info = f" (refs={child.ref_count})" if child.ref_count > 0 else ""
            print(f"{prefix}{connector}{tokens_str}{ref_info}")
            
            extension = "    " if is_child_last else "│   "
            self.visualize(child, prefix + extension, is_child_last, depth + 1)


# Demo: Build a radix tree step by step
tree = SimpleRadixTree()

print("Step 1: Insert 'Hello world' tokens [1, 2, 3, 4]")
tree.insert([1, 2, 3, 4])
tree.visualize()

print("\nStep 2: Insert 'Hello there' tokens [1, 2, 5, 6]")
tree.insert([1, 2, 5, 6])
tree.visualize()

print("\nStep 3: Insert 'Goodbye' tokens [7, 8, 9]")
tree.insert([7, 8, 9])
tree.visualize()

print("\nStep 4: Insert 'Hello world today' tokens [1, 2, 3, 4, 10, 11]")
tree.insert([1, 2, 3, 4, 10, 11])
tree.visualize()

print(f"\nTotal nodes: {tree.total_nodes}")

In [ ]:
# Demonstrate prefix matching

print("Prefix Matching Demo")
print("=" * 50)

test_queries = [
    ([1, 2, 3, 4], "Exact match: 'Hello world'"),
    ([1, 2, 5, 6], "Exact match: 'Hello there'"),
    ([1, 2, 3, 4, 10, 11], "Exact match: 'Hello world today'"),
    ([1, 2, 3, 4, 20, 21], "Prefix match: 'Hello world' + new suffix"),
    ([1, 2, 99], "Partial prefix: 'Hello' + diverge"),
    ([50, 60], "No match: completely new"),
    ([7, 8, 9, 10], "Prefix match: 'Goodbye' + extension"),
]

for query, description in test_queries:
    node, matched = tree.match_prefix(query)
    print(f"\n  Query: {query}")
    print(f"  Description: {description}")
    print(f"  Matched: {matched}/{len(query)} tokens ({matched/len(query)*100:.0f}% cache hit)")
    print(f"  Tokens to recompute: {len(query) - matched}")

## 3. How RadixAttention Stores KV-Cache

### 3.1 The KV-Cache Pool

SGLang maintains a **flat GPU memory pool** for KV-cache:

```
KV-Cache Memory Pool (GPU):
┌────┬────┬────┬────┬────┬────┬────┬────┬────┬────┐
│ S0 │ S1 │ S2 │ S3 │ S4 │ S5 │ S6 │ S7 │ .. │ SN │
└────┴────┴────┴────┴────┴────┴────┴────┴────┴────┘
  ↑    ↑    ↑    ↑    ↑    ↑    ↑    ↑
  └────┴────┘    └────┘    └────┴────┘
  Req A prefix   Req A     Req B
                 suffix
```

Each slot `Si` holds the KV-cache for **one token** across **all layers**.

### 3.2 Radix Tree + KV-Cache Pool

The radix tree nodes hold **indices** into this pool:

```
Radix Tree:                     KV-Cache Pool:
                                ┌────┬────┬────┬────┬────┬────┐
[root]                          │ 0  │ 1  │ 2  │ 3  │ 4  │ 5  │
  └── [sys_prompt]  ────────→   │ K0 │ K1 │ K2 │    │    │    │
       │  kv_indices=[0,1,2]    │ V0 │ V1 │ V2 │    │    │    │
       │                        └────┴────┴────┴────┴────┴────┘
       ├── [user_q1]  ──────→       kv_indices=[3,4]
       │    kv_indices=[3,4]
       │
       └── [user_q2]  ──────→       kv_indices=[5]
            kv_indices=[5]

Request "sys_prompt + user_q1":
  - Cache hit: reuse KV at indices [0,1,2] (system prompt)
  - Compute: only tokens for user_q1, store at [3,4]

Request "sys_prompt + user_q2":
  - Cache hit: reuse KV at indices [0,1,2] (system prompt)
  - Compute: only tokens for user_q2, store at [5]
```

In [ ]:
# Full RadixAttention simulator with KV-cache pool

class KVCachePool:
    """Simulates GPU KV-cache memory pool."""
    
    def __init__(self, max_tokens: int):
        self.max_tokens = max_tokens
        self.used = [False] * max_tokens
        self.num_used = 0
    
    def allocate(self, num_tokens: int) -> List[int]:
        """Allocate slots for tokens. Returns list of indices."""
        indices = []
        for i in range(self.max_tokens):
            if not self.used[i]:
                indices.append(i)
                self.used[i] = True
                self.num_used += 1
                if len(indices) == num_tokens:
                    return indices
        raise MemoryError(f"Cannot allocate {num_tokens} tokens, only {self.available} available")
    
    def free(self, indices: List[int]):
        """Free allocated slots."""
        for idx in indices:
            self.used[idx] = False
            self.num_used -= 1
    
    @property
    def available(self):
        return self.max_tokens - self.num_used
    
    @property
    def utilization(self):
        return self.num_used / self.max_tokens


class RadixAttentionCache:
    """Full RadixAttention implementation with KV-cache pool."""
    
    def __init__(self, max_tokens: int = 1000):
        self.tree = SimpleRadixTree()
        self.pool = KVCachePool(max_tokens)
        self.stats = {
            "total_requests": 0,
            "total_tokens_requested": 0,
            "total_tokens_cached": 0,  # tokens reused from cache
            "total_tokens_computed": 0,  # tokens that needed computation
        }
    
    def process_request(self, token_ids: List[int]) -> dict:
        """Process a request: find cached prefix, allocate new tokens.
        
        Returns dict with cache hit info.
        """
        self.stats["total_requests"] += 1
        self.stats["total_tokens_requested"] += len(token_ids)
        
        # Step 1: Find longest prefix match
        last_node, prefix_len = self.tree.match_prefix(token_ids)
        
        # Step 2: Allocate KV-cache for new tokens
        new_tokens = len(token_ids) - prefix_len
        if new_tokens > 0:
            new_indices = self.pool.allocate(new_tokens)
        else:
            new_indices = []
        
        # Step 3: Insert full sequence into tree
        # (with KV indices for the new portion)
        all_indices = list(range(prefix_len)) + new_indices  # simplified
        self.tree.insert(token_ids, all_indices)
        
        # Update stats
        self.stats["total_tokens_cached"] += prefix_len
        self.stats["total_tokens_computed"] += new_tokens
        
        return {
            "total_tokens": len(token_ids),
            "cached_tokens": prefix_len,
            "new_tokens": new_tokens,
            "cache_hit_rate": prefix_len / len(token_ids) if token_ids else 0,
            "pool_utilization": self.pool.utilization,
        }
    
    def get_stats(self):
        overall_hit_rate = (
            self.stats["total_tokens_cached"] / 
            max(1, self.stats["total_tokens_requested"])
        )
        return {
            **self.stats,
            "overall_cache_hit_rate": overall_hit_rate,
            "pool_utilization": self.pool.utilization,
            "tree_nodes": self.tree.total_nodes,
        }

# Demo: Process multiple requests
cache = RadixAttentionCache(max_tokens=500)

# Simulate system prompt + different questions
system_prompt = list(range(1, 51))  # 50 tokens

requests = [
    ("Request 1 (first, no cache)", system_prompt + [100, 101, 102]),
    ("Request 2 (system prompt cached)", system_prompt + [200, 201, 202, 203]),
    ("Request 3 (system prompt cached)", system_prompt + [300, 301]),
    ("Request 4 (same as req 1, full cache)", system_prompt + [100, 101, 102]),
    ("Request 5 (new system prompt)", list(range(51, 81)) + [400, 401]),
    ("Request 6 (back to first system prompt)", system_prompt + [500, 501, 502]),
]

print("RadixAttention Cache Demo")
print("=" * 70)

for name, tokens in requests:
    result = cache.process_request(tokens)
    bar = "#" * int(result["cache_hit_rate"] * 30) + "." * (30 - int(result["cache_hit_rate"] * 30))
    print(f"\n  {name}")
    print(f"    Tokens: {result['total_tokens']:3d} | Cached: {result['cached_tokens']:3d} | "
          f"New: {result['new_tokens']:3d} | Hit rate: [{bar}] {result['cache_hit_rate']:.0%}")

# Overall stats
stats = cache.get_stats()
print(f"\n{'=' * 70}")
print(f"Overall Statistics:")
print(f"  Total requests: {stats['total_requests']}")
print(f"  Total tokens requested: {stats['total_tokens_requested']}")
print(f"  Tokens served from cache: {stats['total_tokens_cached']}")
print(f"  Tokens computed fresh: {stats['total_tokens_computed']}")
print(f"  Overall cache hit rate: {stats['overall_cache_hit_rate']:.1%}")
print(f"  Pool utilization: {stats['pool_utilization']:.1%}")
print(f"  Tree nodes: {stats['tree_nodes']}")

## 4. Comparison: RadixAttention vs PagedAttention

### 4.1 PagedAttention (vLLM)

```
PagedAttention (block-based, hash-based prefix caching):

Token sequence: [t1, t2, t3, t4, t5, t6, t7, t8, t9]
                 └─ Block 0 ─┘  └─ Block 1 ─┘  └─ B2 ┘
                  hash(t1..t4)   hash(t1..t8)   incomplete

Block Table:
  hash(t1,t2,t3,t4) -> Physical Block 7
  hash(t1,t2,t3,t4,t5,t6,t7,t8) -> Physical Block 12

Pros: Simple, O(1) lookup per block
Cons: Block-level granularity, hash collisions are theoretically possible
      but astronomically unlikely in practice with the hash functions used,
      must opt-in to enable prefix caching
```

### 4.2 RadixAttention (SGLang)

```
RadixAttention (tree-based, automatic prefix sharing):

Token sequence: [t1, t2, t3, t4, t5, t6, t7, t8, t9]

Radix Tree:
  [root]
    └── [t1, t2, t3, t4]  (shared with other sequences starting with t1..t4)
          ├── [t5, t6, t7, t8, t9]  (this sequence)
          └── [t5', t6', ...]  (another sequence diverging at t5)

Pros: Token-level granularity, automatic prefix detection,
      no hash collisions, always enabled
Cons: Tree traversal overhead, more metadata per node
```

In [ ]:
import matplotlib.pyplot as pltimport matplotlib.patches as mpatchesimport numpy as np# Consistent color palette for all diagramsBLUE = '#4A90D9'GREEN = '#27AE60'ORANGE = '#F39C12'RED = '#E74C3C'PURPLE = '#8E44AD'LIGHT_BLUE = '#85C1E9'LIGHT_GREEN = '#82E0AA'LIGHT_ORANGE = '#F8C471'LIGHT_RED = '#F1948A'LIGHT_PURPLE = '#C39BD3'GRAY = '#95A5A6'DARK_GRAY = '#2C3E50'fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))fig.suptitle('RadixAttention vs PagedAttention: Prefix Caching Comparison',             fontsize=14, fontweight='bold', color='#2C3E50', y=0.98)# === Left: PagedAttention (vLLM APC) ===ax1.set_xlim(0, 10)ax1.set_ylim(0, 10)ax1.axis('off')ax1.set_title('PagedAttention (Block-Level Hashing)', fontsize=12, fontweight='bold', color=BLUE, pad=15)# Show blocksblock_colors = [BLUE, BLUE, BLUE, BLUE, GREEN, GREEN, ORANGE, ORANGE]block_labels = ['B0', 'B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7']for i in range(8):    x = 1 + (i % 4) * 2    y = 7.5 - (i // 4) * 1.5    box = mpatches.FancyBboxPatch((x, y), 1.5, 1, boxstyle="round,pad=0.05",                                   facecolor=block_colors[i], alpha=0.7, edgecolor='white', lw=2)    ax1.add_patch(box)    ax1.text(x+0.75, y+0.5, f'{block_labels[i]}\n16 tok', ha='center', va='center',             fontsize=8, color='white', fontweight='bold')# Hash tableax1.text(5, 4.8, 'Hash Table:', fontsize=10, fontweight='bold', color=DARK_GRAY, ha='center')hashes = ['hash(t1..t16) -> B0', 'hash(t1..t32) -> B1', 'hash(t1..t48) -> B2']for i, h in enumerate(hashes):    ax1.text(5, 4.2 - i*0.5, h, fontsize=8, color=DARK_GRAY, ha='center',             fontfamily='monospace')ax1.text(5, 2.5, 'Block-level granularity\n(16 tokens per block)', ha='center',         fontsize=9, color=DARK_GRAY, style='italic',         bbox=dict(boxstyle='round,pad=0.3', facecolor=LIGHT_BLUE, alpha=0.3))ax1.text(5, 1.2, 'Partial block at end\ncannot be cached', ha='center',         fontsize=9, color=RED, fontweight='bold')# === Right: RadixAttention (SGLang) ===ax2.set_xlim(0, 10)ax2.set_ylim(0, 10)ax2.axis('off')ax2.set_title('RadixAttention (Token-Level Radix Tree)', fontsize=12, fontweight='bold', color=GREEN, pad=15)# Draw treedef draw_tree_node(ax, x, y, text, color, w=2.2, h=0.7):    box = mpatches.FancyBboxPatch((x-w/2, y-h/2), w, h, boxstyle="round,pad=0.05",                                   facecolor=color, alpha=0.8, edgecolor='white', lw=2)    ax.add_patch(box)    ax.text(x, y, text, ha='center', va='center', fontsize=7.5, color='white', fontweight='bold')draw_tree_node(ax2, 5, 8.5, "[ROOT]", DARK_GRAY, 1.2, 0.5)draw_tree_node(ax2, 5, 7, "sys prompt\n(50 tokens)", GREEN, 2.5, 0.8)draw_tree_node(ax2, 2.5, 5, "user Q1\n(12 tok)", ORANGE, 2, 0.7)draw_tree_node(ax2, 7.5, 5, "user Q2\n(8 tok)", PURPLE, 2, 0.7)for (x1,y1),(x2,y2) in [((5,8.2),(5,7.4)), ((4,6.6),(2.5,5.35)), ((6,6.6),(7.5,5.35))]:    ax2.annotate('', xy=(x2,y2), xytext=(x1,y1),                arrowprops=dict(arrowstyle='->', color=GRAY, lw=1.5))ax2.text(5, 3.5, 'Token-level granularity\n(any prefix length)', ha='center',         fontsize=9, color=DARK_GRAY, style='italic',         bbox=dict(boxstyle='round,pad=0.3', facecolor=LIGHT_GREEN, alpha=0.3))ax2.text(5, 2.2, 'Automatic prefix detection\nNo hash collisions\nAlways enabled', ha='center',         fontsize=9, color=GREEN, fontweight='bold')# Comparison footerfig.text(0.5, 0.02,         'Key Difference: Radix tree handles ANY prefix length naturally, while block hashing requires full blocks',         ha='center', fontsize=10, color=DARK_GRAY, style='italic')plt.tight_layout(rect=[0, 0.05, 1, 0.95])plt.savefig("radix_vs_paged.png", dpi=150, bbox_inches='tight', facecolor='white')plt.show()

In [ ]:
# Side-by-side comparison simulation

class PagedAttentionCache:
    """Simplified PagedAttention with block-based prefix caching."""
    
    def __init__(self, block_size=16, max_blocks=100):
        self.block_size = block_size
        self.max_blocks = max_blocks
        self.block_table = {}  # hash -> block_id
        self.next_block_id = 0
        self.stats = {"hits": 0, "misses": 0, "total_tokens": 0}
    
    def _hash_block(self, tokens: tuple) -> int:
        return hash(tokens)
    
    def process_request(self, token_ids: List[int]) -> dict:
        self.stats["total_tokens"] += len(token_ids)
        
        cached_tokens = 0
        new_tokens = 0
        
        # Process blocks
        for i in range(0, len(token_ids), self.block_size):
            block_end = min(i + self.block_size, len(token_ids))
            
            if block_end - i < self.block_size:
                # Incomplete block, can't cache
                new_tokens += block_end - i
                continue
            
            # Hash includes all tokens up to this block
            prefix = tuple(token_ids[:block_end])
            block_hash = self._hash_block(prefix)
            
            if block_hash in self.block_table:
                cached_tokens += self.block_size
                self.stats["hits"] += 1
            else:
                new_tokens += self.block_size
                self.block_table[block_hash] = self.next_block_id
                self.next_block_id += 1
                self.stats["misses"] += 1
        
        return {
            "total_tokens": len(token_ids),
            "cached_tokens": cached_tokens,
            "new_tokens": new_tokens,
            "cache_hit_rate": cached_tokens / len(token_ids) if token_ids else 0,
        }

# Compare on the same workload
system_prompt = list(range(1, 65))  # 64 tokens (4 blocks of 16)

workload = [
    system_prompt + list(range(100, 120)),   # 84 tokens
    system_prompt + list(range(200, 215)),   # 79 tokens
    system_prompt + list(range(100, 120)),   # repeat of first
    system_prompt + list(range(300, 310)),   # 74 tokens
    system_prompt + list(range(100, 125)),   # slightly longer version of first
]

radix_cache = RadixAttentionCache(max_tokens=2000)
paged_cache = PagedAttentionCache(block_size=16, max_blocks=200)

print("RadixAttention vs PagedAttention Comparison")
print("=" * 80)
print(f"System prompt: {len(system_prompt)} tokens")
print(f"PagedAttention block size: 16 tokens")
print(f"\n{'Request':<12s} {'Tokens':<8s} {'RadixAtt cached':<18s} {'PagedAtt cached':<18s}")
print("-" * 80)

for i, tokens in enumerate(workload):
    radix_result = radix_cache.process_request(tokens)
    paged_result = paged_cache.process_request(tokens)
    
    print(f"Req {i+1:<8d} {len(tokens):<8d} "
          f"{radix_result['cached_tokens']:3d}/{radix_result['total_tokens']:3d} "
          f"({radix_result['cache_hit_rate']:.0%}){'':8s}"
          f"{paged_result['cached_tokens']:3d}/{paged_result['total_tokens']:3d} "
          f"({paged_result['cache_hit_rate']:.0%})")

radix_stats = radix_cache.get_stats()
print(f"\nOverall RadixAttention hit rate: {radix_stats['overall_cache_hit_rate']:.1%}")
paged_total = paged_cache.stats["hits"] + paged_cache.stats["misses"]
paged_hit_rate = paged_cache.stats["hits"] / max(1, paged_total)
print(f"Overall PagedAttention hit rate: {paged_hit_rate:.1%} (block-level)")

## 5. LRU Eviction

When GPU memory is full, SGLang's RadixCache evicts **least-recently-used leaf nodes**:

```python
# sglang/srt/mem_cache/radix_cache.py — eviction logic (simplified)

class RadixCache:
    def evict(self, num_tokens_to_free: int):
        """Evict LRU leaf nodes to free memory."""
        freed = 0
        
        while freed < num_tokens_to_free:
            # Find the LRU leaf node (oldest last_access)
            leaf = self._find_lru_leaf()
            
            if leaf is None or leaf.ref_count > 0:
                # Can't evict: no leaves or all are in use
                raise MemoryError("Cannot evict enough tokens")
            
            # Free the KV-cache slots
            self.token_pool.free(leaf.kv_indices)
            freed += len(leaf.kv_indices)
            
            # Remove leaf from tree
            parent = leaf.parent
            del parent.children[leaf.token_ids[0]]
            
            # If parent now has only one child, merge them
            if len(parent.children) == 1 and parent != self.root:
                self._merge_with_child(parent)
    
    def _find_lru_leaf(self):
        """Find leaf node with oldest last_access time."""
        oldest_leaf = None
        oldest_time = float('inf')
        
        # BFS to find all leaves
        queue = [self.root]
        while queue:
            node = queue.pop(0)
            if node.is_leaf and node != self.root:
                if node.last_access < oldest_time and node.ref_count == 0:
                    oldest_time = node.last_access
                    oldest_leaf = node
            queue.extend(node.children.values())
        
        return oldest_leaf
```

In [ ]:
import matplotlib.pyplot as pltimport matplotlib.patches as mpatchesimport numpy as np# Consistent color palette for all diagramsBLUE = '#4A90D9'GREEN = '#27AE60'ORANGE = '#F39C12'RED = '#E74C3C'PURPLE = '#8E44AD'LIGHT_BLUE = '#85C1E9'LIGHT_GREEN = '#82E0AA'LIGHT_ORANGE = '#F8C471'LIGHT_RED = '#F1948A'LIGHT_PURPLE = '#C39BD3'GRAY = '#95A5A6'DARK_GRAY = '#2C3E50'fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))fig.suptitle('LRU Eviction in Radix Tree', fontsize=14, fontweight='bold', color='#2C3E50')def draw_tree(ax, title, evicted_node=None):    ax.set_xlim(0, 10)    ax.set_ylim(0, 8)    ax.axis('off')    ax.set_title(title, fontsize=11, fontweight='bold', pad=10)    nodes = [        (5, 7, "[ROOT]", DARK_GRAY, None),        (5, 5.5, "sys_prompt\nt=100", BLUE, "t=100"),        (2.5, 3.5, "user_Q1\nt=50", GREEN, "t=50"),        (5, 3.5, "user_Q2\nt=90", ORANGE, "t=90"),        (7.5, 3.5, "user_Q3\nt=80", PURPLE, "t=80"),        (1.5, 1.5, "resp_A\nt=30", RED, "t=30"),        (3.5, 1.5, "resp_B\nt=60", GREEN, "t=60"),    ]    for x, y, label, color, timestamp in nodes:        if evicted_node and label.startswith(evicted_node):            color = GRAY            alpha = 0.3        else:            alpha = 0.85        box = mpatches.FancyBboxPatch((x-1, y-0.4), 2, 0.8, boxstyle="round,pad=0.05",                                       facecolor=color, alpha=alpha, edgecolor='white', lw=2)        ax.add_patch(box)        ax.text(x, y, label, ha='center', va='center', fontsize=8, color='white', fontweight='bold')    edges = [((5,6.6),(5,5.9)), ((4,5.1),(2.5,3.9)), ((5,5.1),(5,3.9)),             ((6,5.1),(7.5,3.9)), ((2,3.1),(1.5,1.9)), ((3,3.1),(3.5,1.9))]    for (x1,y1),(x2,y2) in edges:        if evicted_node == "resp_A" and (x2 == 1.5 and y2 == 1.9):            continue        ax.annotate('', xy=(x2,y2), xytext=(x1,y1),                    arrowprops=dict(arrowstyle='->', color=GRAY, lw=1.2))# Before evictiondraw_tree(ax1, 'Before Eviction')ax1.annotate('LRU leaf\n(oldest: t=30)', xy=(1.5, 1.1), xytext=(1.5, 0.2),             fontsize=9, color=RED, fontweight='bold', ha='center',             arrowprops=dict(arrowstyle='->', color=RED, lw=2))# After evictiondraw_tree(ax2, 'After Eviction', evicted_node="resp_A")ax2.text(1.5, 0.5, 'EVICTED\n(freed KV slots)', ha='center', fontsize=9,         color=RED, fontweight='bold',         bbox=dict(boxstyle='round,pad=0.2', facecolor=LIGHT_RED, alpha=0.3))plt.tight_layout()plt.savefig("lru_eviction.png", dpi=150, bbox_inches='tight', facecolor='white')plt.show()

In [ ]:
# Demonstrate LRU eviction in the radix tree

class RadixCacheWithEviction(RadixAttentionCache):
    """RadixAttention cache with LRU eviction."""
    
    def evict_lru(self, num_tokens: int):
        """Evict least-recently-used leaf nodes."""
        freed = 0
        evicted_nodes = []
        
        while freed < num_tokens:
            leaf = self._find_lru_leaf()
            if leaf is None:
                print(f"    Cannot evict more! Only freed {freed} tokens")
                break
            
            evicted_nodes.append(list(leaf.token_ids))
            freed += len(leaf.token_ids)
            
            # Remove from tree
            if leaf.parent:
                for key, child in list(leaf.parent.children.items()):
                    if child is leaf:
                        del leaf.parent.children[key]
                        break
            self.tree.total_nodes -= 1
        
        return freed, evicted_nodes
    
    def _find_lru_leaf(self):
        """Find the LRU leaf node."""
        oldest = None
        oldest_time = float('inf')
        
        def search(node):
            nonlocal oldest, oldest_time
            if node.is_leaf and node != self.tree.root:
                if node.ref_count == 0 and node.last_access < oldest_time:
                    oldest = node
                    oldest_time = node.last_access
            for child in node.children.values():
                search(child)
        
        search(self.tree.root)
        return oldest

# Demo eviction
cache = RadixCacheWithEviction(max_tokens=100)

# Insert several sequences
import time
sequences = [
    list(range(1, 21)),    # 20 tokens, oldest
    list(range(1, 16)) + list(range(50, 60)),  # shared prefix
    list(range(30, 55)),   # 25 tokens
    list(range(60, 80)),   # 20 tokens, newest
]

print("Cache State Before Eviction")
print("=" * 50)

for seq in sequences:
    result = cache.process_request(seq)
    time.sleep(0.01)  # Ensure different timestamps

cache.tree.visualize()
print(f"\nPool utilization: {cache.pool.utilization:.1%}")

# Evict
print("\n\nEvicting 30 tokens (LRU)...")
freed, evicted = cache.evict_lru(30)
print(f"Freed: {freed} tokens")
print(f"Evicted nodes: {evicted}")

print("\nCache State After Eviction")
print("=" * 50)
cache.tree.visualize()

## 6. Cache Hit Scenarios

### 6.1 System Prompt Sharing

```
Most common scenario: all chat requests share a system prompt.

Tree after multiple requests:
[root]
  └── [system_prompt: "You are a helpful assistant..."]  (50 tokens)
       ├── [user_q1: "What is AI?"]  (5 tokens)
       ├── [user_q2: "Tell me about Python"]  (6 tokens)
       ├── [user_q3: "Explain transformers"]  (4 tokens)
       └── ...

Cache hit rate: 50/(50+5) = 91% per request!
```

### 6.2 Multi-Turn Conversations

```
Turn 1: [sys | user_q1 | assistant_a1]
Turn 2: [sys | user_q1 | assistant_a1 | user_q2 | assistant_a2]
Turn 3: [sys | user_q1 | assistant_a1 | user_q2 | assistant_a2 | user_q3]

Tree:
[root]
  └── [sys | user_q1 | assistant_a1]  (cached from turn 1)
       └── [user_q2 | assistant_a2]  (cached from turn 2)
            └── [user_q3]  (new in turn 3)

Turn 3 cache hit: (sys + q1 + a1 + q2 + a2) / total = very high!
```

### 6.3 Few-Shot Examples

```
Request pattern:
  [system | example1 | example2 | example3 | new_input]

All requests share [system | example1 | example2 | example3]
Only [new_input] varies.

Tree:
[root]
  └── [system | example1 | example2 | example3]  (200 tokens)
       ├── [input_A]  (10 tokens)
       ├── [input_B]  (12 tokens)
       └── [input_C]  (8 tokens)

Cache hit rate: 200/210 = 95%+!
```

In [ ]:
# Simulate real-world cache hit scenarios

import random

def simulate_scenario(name, requests, max_tokens=5000):
    """Run a workload scenario and report cache statistics."""
    cache = RadixAttentionCache(max_tokens=max_tokens)
    results = []
    
    for tokens in requests:
        result = cache.process_request(tokens)
        results.append(result)
    
    stats = cache.get_stats()
    return stats, results

# Scenario 1: System prompt sharing
sys_prompt = list(range(1, 101))  # 100 tokens
scenario1_requests = []
for i in range(20):
    user_tokens = list(range(1000 + i*20, 1000 + i*20 + random.randint(5, 15)))
    scenario1_requests.append(sys_prompt + user_tokens)

# Scenario 2: Multi-turn conversation
scenario2_requests = []
conversation = list(range(1, 51))  # system prompt
for turn in range(10):
    user_q = list(range(2000 + turn*30, 2000 + turn*30 + 15))
    assistant_a = list(range(3000 + turn*50, 3000 + turn*50 + 30))
    conversation = conversation + user_q + assistant_a
    scenario2_requests.append(list(conversation))  # copy

# Scenario 3: Few-shot examples
examples = list(range(1, 201))  # 200 tokens of examples
scenario3_requests = []
for i in range(30):
    new_input = list(range(5000 + i*10, 5000 + i*10 + random.randint(5, 10)))
    scenario3_requests.append(examples + new_input)

# Scenario 4: No sharing (worst case)
scenario4_requests = []
for i in range(20):
    scenario4_requests.append(list(range(i*100, i*100 + random.randint(50, 80))))

scenarios = [
    ("System Prompt (100 tok shared)", scenario1_requests),
    ("Multi-Turn (growing context)", scenario2_requests),
    ("Few-Shot (200 tok examples)", scenario3_requests),
    ("No Sharing (worst case)", scenario4_requests),
]

print("Cache Hit Rate by Scenario")
print("=" * 75)
print(f"{'Scenario':<35s} {'Requests':>8s} {'Hit Rate':>10s} {'Saved Compute':>15s}")
print("-" * 75)

for name, requests in scenarios:
    stats, results = simulate_scenario(name, requests)
    saved_compute = stats['total_tokens_cached'] / max(1, stats['total_tokens_requested'])
    print(f"{name:<35s} {stats['total_requests']:8d} "
          f"{stats['overall_cache_hit_rate']:9.1%} "
          f"{stats['total_tokens_cached']:8d} tokens")

## 7. Source Code Walkthrough: `RadixCache`

Let's look at the actual SGLang `RadixCache` implementation:

```python
# sglang/srt/mem_cache/radix_cache.py (simplified)

class TreeNode:
    """A node in the radix tree."""
    
    counter = 0  # Global counter for unique IDs
    
    def __init__(self):
        self.children = {}           # token_id -> TreeNode
        self.parent = None
        self.key = None              # Token ID on the edge to this node
        self.value = None            # KV-cache index in the pool
        self.lock_ref = 0            # Number of active references
        self.last_access_time = 0    # For LRU eviction
        self.id = TreeNode.counter   # Unique node ID
        TreeNode.counter += 1


class RadixCache:
    """Radix tree-based KV-cache manager."""
    
    def __init__(self, req_to_token_pool, token_to_kv_pool, disable=False):
        self.root_node = TreeNode()  # Root of the radix tree
        self.root_node.key = []
        self.root_node.value = []
        self.root_node.lock_ref = 1  # Root is always locked
        
        self.req_to_token_pool = req_to_token_pool  # Manages request -> token mapping
        self.token_to_kv_pool = token_to_kv_pool    # The actual KV-cache GPU memory
        self.disable = disable
        
        self.evictable_size_ = 0  # Tokens available for eviction
    
    def match_prefix(self, key: List[int]) -> Tuple[TreeNode, int]:
        """Find the longest prefix match.
        
        Args:
            key: List of token IDs to match
            
        Returns:
            (last_node, prefix_length): The deepest matching node
            and number of matched tokens
        """
        if self.disable:
            return self.root_node, 0
        
        node = self.root_node
        matched_len = 0
        
        while matched_len < len(key):
            token = key[matched_len]
            if token in node.children:
                child = node.children[token]
                child_key = child.key
                
                # Match as many tokens as possible on this edge
                prefix_len = _match_length(
                    child_key, key[matched_len:]
                )
                matched_len += prefix_len
                
                if prefix_len < len(child_key):
                    # Partial match on edge
                    break
                
                node = child
            else:
                break
        
        return node, matched_len
    
    def insert(self, key: List[int], value: List[int] = None):
        """Insert a token sequence into the tree.
        
        If the sequence shares a prefix with existing entries,
        only the new suffix is added.
        """
        if self.disable:
            return 0
        
        node, matched_len = self.match_prefix(key)
        
        if matched_len == len(key):
            return matched_len  # Already fully in tree
        
        # Need to handle potential edge splitting
        # and insert remaining tokens
        # ... (edge splitting logic) ...
        
        return matched_len
    
    def cache_finished_req(self, req):
        """Cache the KV-cache from a finished request.
        
        Called when a request completes generation.
        Inserts the full sequence (prompt + generation) into the tree.
        """
        token_ids = req.origin_input_ids + req.output_ids
        kv_indices = req.req_pool_indices  # GPU memory locations
        
        # Insert into tree, linking to GPU KV-cache
        self.insert(token_ids, kv_indices)
        
        # Decrease reference count (request no longer active)
        self.dec_lock_ref(req.last_node)
    
    def evict(self, num_tokens: int):
        """Evict LRU nodes to free GPU memory."""
        if self.disable:
            return
        
        leaves = self._collect_leaves()
        # Sort by last access time (LRU first)
        leaves.sort(key=lambda x: x.last_access_time)
        
        freed = 0
        for leaf in leaves:
            if freed >= num_tokens:
                break
            if leaf.lock_ref == 0:  # Not in use
                freed += len(leaf.value)
                self._remove_leaf(leaf)
        
        return freed
```

In [ ]:
# Key insight: the match_prefix function in detail

def _match_length(edge_tokens, query_tokens):
    """Find the common prefix length between two token sequences."""
    min_len = min(len(edge_tokens), len(query_tokens))
    for i in range(min_len):
        if edge_tokens[i] != query_tokens[i]:
            return i
    return min_len

# Demo the match_prefix algorithm step by step
print("match_prefix Algorithm Walkthrough")
print("=" * 60)

# Build a tree
tree = SimpleRadixTree()
tree.insert([10, 20, 30, 40, 50])  # "Hello world how"
tree.insert([10, 20, 30, 60, 70])  # "Hello world what"
tree.insert([10, 20, 80, 90])      # "Hello there"

print("Tree structure:")
tree.visualize()

# Trace a match
query = [10, 20, 30, 40, 55, 65]
print(f"\nQuery: {query}")
print("\nStep-by-step matching:")

node = tree.root
pos = 0
step = 0

while pos < len(query):
    step += 1
    first_token = query[pos]
    print(f"\n  Step {step}: pos={pos}, looking for token {first_token}")
    
    if first_token not in node.children:
        print(f"    No child starting with {first_token}. Stop.")
        break
    
    child = node.children[first_token]
    print(f"    Found child with edge: {list(child.token_ids)}")
    
    match_len = _match_length(child.token_ids, query[pos:])
    print(f"    Match length: {match_len} out of {len(child.token_ids)} edge tokens")
    
    if match_len < len(child.token_ids):
        print(f"    Partial match! Matched {pos + match_len} total tokens.")
        pos += match_len
        break
    
    print(f"    Full edge match! Continue down tree.")
    node = child
    pos += match_len

print(f"\nResult: matched {pos}/{len(query)} tokens")
print(f"Cache hit rate: {pos/len(query):.0%}")
print(f"Tokens to compute: {len(query) - pos}")

## 8. The Connection to Scheduling

RadixAttention doesn't just cache — it also informs **scheduling decisions**:

### Longest Prefix Match (LPM) Scheduling

```python
# sglang/srt/managers/scheduler.py — LPM scheduling (simplified)

def schedule_with_lpm(waiting_queue, radix_cache):
    """Schedule requests prioritizing those with longest cache matches."""
    
    # Score each waiting request by its cache hit potential
    scored_requests = []
    for req in waiting_queue:
        _, prefix_len = radix_cache.match_prefix(req.input_ids)
        score = prefix_len / len(req.input_ids)
        scored_requests.append((score, req))
    
    # Sort by score (highest first = most cache reuse)
    scored_requests.sort(key=lambda x: x[0], reverse=True)
    
    # Schedule top-scoring requests
    batch = []
    for score, req in scored_requests:
        if can_add_to_batch(req):
            batch.append(req)
    
    return batch
```

**Why LPM matters**: By scheduling requests with the best cache hits first, we:
1. Minimize total compute (reuse more cached KV)
2. Maximize throughput (process more tokens per forward pass)
3. Keep the radix tree hot (popular prefixes stay cached)

In [ ]:
# Demonstrate LPM vs FCFS scheduling impact

def simulate_scheduling(policy, requests, cache):
    """Simulate scheduling with different policies."""
    total_compute = 0
    
    if policy == "lpm":
        # Score by prefix match and sort
        scored = []
        for req in requests:
            _, prefix_len = cache.tree.match_prefix(req)
            scored.append((prefix_len / len(req), req))
        scored.sort(key=lambda x: x[0], reverse=True)
        ordered = [req for _, req in scored]
    else:
        ordered = requests  # FCFS
    
    results = []
    for req in ordered:
        result = cache.process_request(req)
        total_compute += result["new_tokens"]
        results.append(result)
    
    return total_compute, results

# Create a workload with mixed cache hit potential
system_a = list(range(1, 51))     # System prompt A (50 tokens)
system_b = list(range(100, 151))  # System prompt B (50 tokens)

workload = [
    # Mix of requests using both system prompts
    system_a + [200, 201, 202],       # A + short query
    system_b + [300, 301, 302, 303],  # B + query
    system_a + [400, 401],            # A + short query (will benefit from cache)
    system_b + [500, 501, 502],       # B + query (will benefit from cache)
    system_a + [600, 601, 602, 603],  # A + query (will benefit from cache)
    list(range(700, 730)),             # Completely new (no cache benefit)
]

print("Scheduling Policy Impact on Cache Efficiency")
print("=" * 60)

for policy in ["fcfs", "lpm"]:
    cache = RadixAttentionCache(max_tokens=5000)
    # Pre-populate cache with system_a
    cache.process_request(system_a + [999])
    
    total_compute, results = simulate_scheduling(policy, workload, cache)
    total_requested = sum(len(r) for r in workload)
    
    print(f"\n  Policy: {policy.upper()}")
    print(f"  Total tokens requested: {total_requested}")
    print(f"  Total compute needed: {total_compute}")
    print(f"  Compute saved: {total_requested - total_compute} tokens ({(total_requested - total_compute)/total_requested:.0%})")

## 9. Memory Pool Implementation

The RadixCache works with a **GPU memory pool** that pre-allocates KV-cache slots:

```python
# sglang/srt/mem_cache/memory_pool.py (simplified)

class TokenToKVPool:
    """Pre-allocated GPU memory pool for KV-cache.
    
    Allocates a large contiguous tensor on GPU and manages
    individual token slots within it.
    """
    
    def __init__(self, total_tokens, num_layers, num_heads, head_dim, dtype):
        # Pre-allocate GPU memory
        # Shape: [total_tokens, num_layers, 2, num_heads, head_dim]
        #         2 = K and V
        self.kv_data = torch.empty(
            (total_tokens, num_layers, 2, num_heads, head_dim),
            dtype=dtype,
            device="cuda"
        )
        
        # Free list (stack of available indices)
        self.free_slots = list(range(total_tokens))
        self.total_tokens = total_tokens
    
    def alloc(self, num_tokens: int) -> torch.Tensor:
        """Allocate slots, returns tensor of indices."""
        if len(self.free_slots) < num_tokens:
            return None  # Out of memory
        
        indices = self.free_slots[:num_tokens]
        self.free_slots = self.free_slots[num_tokens:]
        return torch.tensor(indices, dtype=torch.int32, device="cuda")
    
    def free(self, indices: torch.Tensor):
        """Return slots to the pool."""
        self.free_slots.extend(indices.cpu().tolist())
    
    @property
    def available(self):
        return len(self.free_slots)
```

In [ ]:
# Calculate memory requirements for the KV-cache pool

def calculate_kv_pool_memory(
    model_name: str,
    num_layers: int,
    num_kv_heads: int,
    head_dim: int,
    dtype_bytes: int,  # 2 for FP16
    gpu_memory_gb: float,
    model_size_gb: float,
    mem_fraction: float = 0.88,
):
    """Calculate how many tokens can fit in the KV-cache pool."""
    
    # Memory per token
    # 2 (K+V) * num_layers * num_kv_heads * head_dim * dtype_bytes
    bytes_per_token = 2 * num_layers * num_kv_heads * head_dim * dtype_bytes
    
    # Available memory
    total_gpu_bytes = gpu_memory_gb * 1024**3
    model_bytes = model_size_gb * 1024**3
    available_bytes = total_gpu_bytes * mem_fraction - model_bytes
    
    # Max tokens
    max_tokens = int(available_bytes / bytes_per_token)
    
    return {
        "model": model_name,
        "bytes_per_token": bytes_per_token,
        "available_gb": available_bytes / 1024**3,
        "max_tokens": max_tokens,
        "max_tokens_k": max_tokens / 1000,
    }

models = [
    ("Llama-3.1-8B", 32, 8, 128, 2, 80, 16),     # A100 80GB
    ("Llama-3.1-70B (TP=4)", 80, 2, 128, 2, 80, 35),  # per GPU with TP=4
    ("Qwen2.5-7B", 28, 4, 128, 2, 80, 14),
    ("Mixtral-8x7B", 32, 8, 128, 2, 80, 24),     # per GPU with TP=2
]

print("KV-Cache Pool Memory Analysis")
print("=" * 80)
print(f"{'Model':<25s} {'B/token':>8s} {'Avail GB':>10s} {'Max Tokens':>12s} {'Max Seqs*':>10s}")
print("-" * 80)

for name, layers, kv_heads, head_dim, dtype, gpu_gb, model_gb in models:
    result = calculate_kv_pool_memory(name, layers, kv_heads, head_dim, dtype, gpu_gb, model_gb)
    max_seqs = result["max_tokens"] // 4096  # Assuming 4K avg context
    print(f"{result['model']:<25s} {result['bytes_per_token']:>8,d} "
          f"{result['available_gb']:>9.1f}  {result['max_tokens']:>11,d} {max_seqs:>10,d}")

print("\n* Max concurrent sequences assuming 4K average context length")

## 10. Summary

### Key Takeaways

1. **RadixAttention** uses a radix tree to automatically detect and share KV-cache prefixes
2. **The radix tree** (Patricia trie) stores token sequences with shared prefix nodes
3. **match_prefix()** traverses the tree to find the longest cached prefix
4. **LRU eviction** removes leaf nodes when GPU memory is full
5. **LPM scheduling** prioritizes requests with the best cache hit potential
6. **Cache hit rates** of 90%+ are common for system prompt sharing and multi-turn conversations
7. **vs PagedAttention**: RadixAttention provides automatic, token-level prefix caching vs. block-level, opt-in

### Next Chapter

Chapter 5.4 will dive into the **SGLang Scheduler** — how it manages batching, scheduling policies, and interacts with RadixCache.

## Exercises

### Exercise 1: Implement Radix Tree with Eviction
Extend the `SimpleRadixTree` class to support:
- Reference counting (inc/dec on request start/end)
- LRU eviction that only evicts unreferenced leaves
- Tree compaction after eviction (merge single-child nodes)

### Exercise 2: Cache Hit Analysis
Given a workload trace of 1000 requests (generate your own), measure:
- Average cache hit rate over time (does it improve?)
- Memory utilization over time
- Impact of cache size on hit rate

### Exercise 3: Advanced Eviction Policy
Implement an eviction policy that considers both recency AND frequency:
- LFU (Least Frequently Used)
- LRU-K (considers K-th most recent access)
Compare with simple LRU on a realistic workload.

### Exercise 4: Visualize the Tree
Create a graphical visualization of the radix tree using matplotlib or graphviz that shows:
- Node sizes (proportional to stored tokens)
- Reference counts (color coding)
- Last access time (opacity)

In [ ]:
# Exercise 1 starter code

class RadixTreeWithEviction:
    """TODO: Implement radix tree with reference counting and LRU eviction."""
    
    def __init__(self, max_tokens: int):
        self.root = RadixNode()
        self.max_tokens = max_tokens
        self.current_tokens = 0
    
    def inc_ref(self, node: RadixNode):
        """Increment reference count for node and all ancestors."""
        # TODO: Implement
        pass
    
    def dec_ref(self, node: RadixNode):
        """Decrement reference count for node and all ancestors."""
        # TODO: Implement
        pass
    
    def evict_if_needed(self, required_tokens: int):
        """Evict LRU leaves until enough space is available."""
        # TODO: Implement
        # 1. Find all unreferenced (ref_count == 0) leaf nodes
        # 2. Sort by last_access time
        # 3. Evict oldest until required_tokens are freed
        # 4. Compact tree (merge single-child nodes)
        pass
    
    def compact(self, node: RadixNode):
        """Merge single-child internal nodes with their child."""
        # TODO: Implement
        pass

print("Exercise 1: Implement the methods above!")
print("Test with:")
print("  tree = RadixTreeWithEviction(max_tokens=100)")
print("  # Insert sequences, check ref counts, trigger eviction")